# Chapter 124: Attention Rollout Trading

This notebook demonstrates how to use Attention Rollout for interpretable trading predictions.

## Contents
1. Introduction to Attention Rollout
2. Loading Market Data
3. Building a Trading Transformer
4. Computing Attention Rollout
5. Analyzing Trading Patterns
6. Backtesting with Attention Analysis

## 1. Setup and Imports

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from attention_rollout import AttentionRollout, TradingAttentionRollout
from model import TradingTransformer
from data_loader import load_stock_data, load_bybit_data, prepare_features
from backtest import AttentionBacktester, print_backtest_report
from visualization import (
    plot_attention_rollout,
    plot_input_attribution,
    plot_attention_comparison,
    plot_equity_curve
)

print("All modules loaded successfully!")

## 2. Loading Market Data

We'll load data from both stock markets and cryptocurrency exchanges.

In [ ]:
# Load cryptocurrency data from Bybit
print("Loading BTC/USDT data from Bybit...")
crypto_df = load_bybit_data(symbol="BTCUSDT", interval="D", limit=200)
print(f"Loaded {len(crypto_df)} candles")
crypto_df.head()

In [ ]:
# Prepare features for model
lookback = 20
feature_cols = ["open", "high", "low", "close", "volume"]

X, y = prepare_features(crypto_df, feature_columns=feature_cols, lookback=lookback)
print(f"Feature shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Label distribution: {np.bincount(y.astype(int))}")

## 3. Building the Trading Transformer Model

In [ ]:
# Initialize the model
model = TradingTransformer(
    input_dim=X.shape[2],
    d_model=64,
    n_heads=4,
    n_layers=3,
    dropout=0.1,
    max_seq_len=lookback
)

print("Model architecture:")
print(model)

In [ ]:
# Test forward pass
x_sample = torch.FloatTensor(X[:1])
logits, attention_maps = model(x_sample)

print(f"Input shape: {x_sample.shape}")
print(f"Output logits shape: {logits.shape}")
print(f"Number of attention maps: {len(attention_maps)}")
print(f"Attention map shape: {attention_maps[0].shape}")

## 4. Computing Attention Rollout

In [ ]:
# Initialize Attention Rollout
rollout = TradingAttentionRollout(
    model,
    attention_layer_name="attn",
    head_fusion="mean",
    feature_names=feature_cols
)

# Compute rollout for a sample
rollout_matrix = rollout.compute_rollout(x_sample)
print(f"Rollout matrix shape: {rollout_matrix.shape}")

In [ ]:
# Get input attribution scores
attribution = rollout.get_input_attribution(x_sample)

# Create timestamp labels
timestamps = [f"t-{i}" for i in range(lookback-1, -1, -1)]

# Display attribution
print("Input Attribution Scores:")
for i, (ts, score) in enumerate(zip(timestamps, attribution)):
    bar = '#' * int(score * 50)
    print(f"{ts:>5}: {bar:50} {score:.4f}")

In [ ]:
# Visualize attribution
plot_input_attribution(attribution, timestamps, "Input Attribution for Trading Prediction")

## 5. Analyzing Trading Patterns

In [ ]:
# Analyze temporal importance
temporal_importance = rollout.analyze_temporal_importance(x_sample, timestamps)
print("Temporal Importance:")
for ts, score in sorted(temporal_importance.items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f"  {ts}: {score:.4f}")

In [ ]:
# Detect attention regime
regime = rollout.detect_attention_regime(x_sample)
print(f"Detected Regime: {regime}")

regime_explanations = {
    "momentum": "Model focuses on recent price action (trend-following)",
    "mean_reversion": "Model focuses on historical patterns (mean-reversion)",
    "mixed": "Model uses balanced attention across time periods"
}
print(f"Interpretation: {regime_explanations[regime]}")

## 6. Backtesting with Attention Analysis

In [ ]:
# Prepare prices for backtest
prices = crypto_df["close"].values[-len(X)-1:-1]

# Split data
train_size = int(len(X) * 0.7)
X_test = X[train_size:]
prices_test = prices[train_size:]

print(f"Test set size: {len(X_test)} samples")

In [ ]:
# Run backtest with attention analysis
backtester = AttentionBacktester(
    model=model,
    attention_rollout=rollout,
    initial_capital=100000.0,
    transaction_cost=0.001
)

result = backtester.run_backtest(X_test, prices_test, threshold=0.5)
print_backtest_report(result)

In [ ]:
# Plot equity curve
plot_equity_curve(result.equity_curve, title="Strategy Equity Curve")

In [ ]:
# Compare winning vs losing trade attention patterns
if result.attention_analysis:
    winning_attn = np.array(result.attention_analysis.get("avg_winning_attention", []))
    losing_attn = np.array(result.attention_analysis.get("avg_losing_attention", []))
    
    if len(winning_attn) > 0 and len(losing_attn) > 0:
        plot_attention_comparison(
            winning_attn,
            losing_attn,
            timestamps
        )

## Conclusion

In this notebook, we demonstrated:

1. **Attention Rollout** - How to compute and visualize attention flow through transformer layers
2. **Input Attribution** - Which time periods most influence trading predictions
3. **Regime Detection** - Identifying whether the model uses momentum or mean-reversion patterns
4. **Backtesting** - Evaluating trading performance with attention pattern analysis

Key takeaways:
- Attention rollout provides interpretability for black-box trading models
- Different attention patterns (momentum vs mean-reversion) indicate different trading strategies
- Comparing winning and losing trade patterns reveals model behavior insights